In [1]:
import os
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd
import cv2

from google.colab import drive
drive.mount('/content/drive')

os.environ["SM_FRAMEWORK"] = "tf.keras"
import segmentation_models as sm

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Segmentation Models: using `tf.keras` framework.


In [2]:
IMAGE_DIR   = "/content/drive/MyDrive/floodsegnet/data/Image"
MASK_DIR    = "/content/drive/MyDrive/floodsegnet/data/Mask"
METADATA    = "/content/drive/MyDrive/floodsegnet/data/metadata.csv"

metadata = pd.read_csv(METADATA)
metadata.head()

image_paths = [
    os.path.join(IMAGE_DIR, img_name)
    for img_name in metadata["Image"]
]

mask_paths = [
    os.path.join(MASK_DIR, mask_name)
    for mask_name in metadata["Mask"]
]

print(f"Images: {len(image_paths)}, Masks: {len(mask_paths)}")

Images: 290, Masks: 290


In [3]:
from sklearn.model_selection import train_test_split

train_images, val_images, train_masks, val_masks = train_test_split(
    image_paths,
    mask_paths,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(train_images))
print("Validation samples:", len(val_images))

Training samples: 232
Validation samples: 58


In [4]:
print(train_images[:5])
print(train_masks[:5])

['/content/drive/MyDrive/floodsegnet/data/Image/6.jpg', '/content/drive/MyDrive/floodsegnet/data/Image/1007.jpg', '/content/drive/MyDrive/floodsegnet/data/Image/1021.jpg', '/content/drive/MyDrive/floodsegnet/data/Image/1059.jpg', '/content/drive/MyDrive/floodsegnet/data/Image/2024.jpg']
['/content/drive/MyDrive/floodsegnet/data/Mask/6.png', '/content/drive/MyDrive/floodsegnet/data/Mask/1007.png', '/content/drive/MyDrive/floodsegnet/data/Mask/1021.png', '/content/drive/MyDrive/floodsegnet/data/Mask/1059.png', '/content/drive/MyDrive/floodsegnet/data/Mask/2024.png']


In [5]:
IMG_SIZE = 256
BATCH_SIZE = 8

In [ ]:
def load_single(img_path, mask_path):
    # cv2 load
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image.astype(np.float32) / 255.0

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
    mask = (mask > 127).astype(np.float32)
    mask = np.expand_dims(mask, axis=-1)

    return image, mask

# load everything into memory upfront — dataset is only 290 images, fits easily in 32GB RAM
all_images = []
all_masks  = []

for img_p, msk_p in zip(image_paths, mask_paths):
    img, msk = load_single(img_p, msk_p)
    all_images.append(img)
    all_masks.append(msk)

all_images = np.array(all_images)  # (290, 256, 256, 3)
all_masks  = np.array(all_masks)   # (290, 256, 256, 1)

print(f"Images: {all_images.shape}")
print(f"Masks:  {all_masks.shape}")

In [ ]:
# split
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    all_images, all_masks, test_size=0.2, random_state=42
)

In [ ]:
# tf datasets — no map, no numpy_function, no AutoGraph issues
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = train_dataset.shuffle(100).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [ ]:
# sanity check
for images, masks in train_dataset.take(1):
    print(f"Image batch: {images.shape}")  # (8, 256, 256, 3)
    print(f"Mask batch:  {masks.shape}")   # (8, 256, 256, 1)
    print(f"Mask unique: {np.unique(masks.numpy())}")  # [0. 1.]

In [ ]:
for images, masks in train_dataset.take(1):
    print(images.shape)
    print(masks.shape)

In [ ]:
model = sm.Unet(
    backbone_name="resnet34",
    encoder_weights="imagenet",
    classes=1,
    activation="sigmoid"
)

In [ ]:
model.compile(
    optimizer='adam',
    loss=sm.losses.bce_dice_loss,
    metrics=[
        sm.metrics.iou_score,
        sm.metrics.f1_score
    ]
)

In [ ]:
callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        "best_model.keras",
        monitor="val_f1-score",
        mode="max",
        save_best_only=True
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_f1-score",
        mode="max",
        patience=8,
        restore_best_weights=True
    )

]

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=30,
    callbacks=callbacks
)

BATCH_SIZE = 8